# C vs Java (OOP) — Learning Comparison

Goal: understand Java OOP by mapping it to C concepts already known.

**Contents**
1. Counter: struct + functions vs class
2. Shapes: function pointers vs inheritance and polymorphism
3. Java keywords explained (`class`, `public`, `private`, `static`, `void`, `String[] args`, `new`)
4. Summary table
5. Exercises

> Run requirements: `gcc` and JDK 17+ (`java File.java` runs a single file directly).

## 1. Counter: `struct` + functions vs `class`

Same program, two languages. The key point: in C nothing protects `value`, in Java `private` does.

In [1]:
%%writefile counter.c
#include <stdio.h>

typedef struct {
    int value;
} Counter;

void counter_add(Counter *c, int n) {
    c->value += n;
}

int main(void) {
    Counter c = {0};
    counter_add(&c, 5);
    c.value = -100;              /* nothing stops this */
    printf("%d\n", c.value);
    return 0;
}

Writing counter.c


In [2]:
!gcc counter.c -o counter && ./counter

-100


In [3]:
%%writefile Counter.java
public class Counter {
    private int value;               // private: only this class can touch it

    public void add(int n) {
        value += n;
    }

    public int getValue() {
        return value;
    }

    public static void main(String[] args) {
        Counter c = new Counter();
        c.add(5);
        // c.value = -100;           // compile error: value is private
        System.out.println(c.getValue());
    }
}

Writing Counter.java


In [4]:
!java Counter.java

5


**Proof that `private` works.** This version tries to touch `value` from outside; the compiler refuses.

In [5]:
%%writefile Broken.java
class Counter2 {
    private int value;
}

public class Broken {
    public static void main(String[] args) {
        Counter2 c = new Counter2();
        c.value = -100;
    }
}

Writing Broken.java


In [6]:
!java Broken.java 2>&1 | head -5

Broken.java:8: error: value has private access in Counter2
        c.value = -100;
         ^
1 error
error: compilation failed


**Differences**

| | C | Java |
|---|---|---|
| Data and behavior | `struct` and functions are separate | One `class` holds both |
| Call style | `counter_add(&c, 5)` | `c.add(5)` |
| Protection | none, anyone edits `c.value` | `private` blocks outside access |
| Memory | `malloc` / `free` | `new`, garbage collector frees |

## 2. Shapes: function pointers vs inheritance

C builds polymorphism by hand with a function pointer. Java does it with `extends` + `@Override`, and the compiler verifies it.

In [7]:
%%writefile shapes.c
#include <stdio.h>

typedef struct Shape Shape;
struct Shape {
    double a, b;                            /* generic fields: meaning depends on the shape */
    double (*area)(const Shape *s);         /* function pointer, wired by hand */
};

double circle_area(const Shape *s) { return 3.14159 * s->a * s->a; }
double rect_area(const Shape *s)   { return s->a * s->b; }

int main(void) {
    Shape shapes[2] = {
        {2.0, 0.0, circle_area},            /* a = radius */
        {3.0, 4.0, rect_area}               /* a = width, b = height */
    };

    for (int i = 0; i < 2; i++)
        printf("area: %.2f\n", shapes[i].area(&shapes[i]));
    return 0;
}

Writing shapes.c


In [8]:
!gcc shapes.c -o shapes && ./shapes

area: 12.57
area: 12.00


In [9]:
%%writefile Main.java
public class Main {
    public static void main(String[] args) {
        Shape[] shapes = { new Circle(2.0), new Rectangle(3.0, 4.0) };

        for (Shape s : shapes)
            System.out.printf("area: %.2f%n", s.area());
    }
}

abstract class Shape {
    public abstract double area();          // every shape MUST implement this
}

class Circle extends Shape {
    private final double radius;            // named fields, not generic a, b
    public Circle(double radius) { this.radius = radius; }
    @Override public double area() { return Math.PI * radius * radius; }
}

class Rectangle extends Shape {
    private final double width, height;
    public Rectangle(double width, double height) {
        this.width = width;
        this.height = height;
    }
    @Override public double area() { return width * height; }
}

Writing Main.java


In [10]:
!java Main.java

area: 12.57
area: 12.00


**Differences**

1. **Fields:** C uses generic `a, b` (you must remember their meaning). Java gives each class named fields.
2. **Polymorphism:** C sets a function pointer manually. Java uses `extends` + `@Override`.
3. **Safety:** forgetting `area` in C crashes at runtime. Forgetting `area()` in Java is a compile error.
4. **Calling:** C passes the object (`area(&shapes[i])`). Java passes it implicitly (`s.area()`).

## 3. Java keywords explained

### `class`
A blueprint: a C `struct` plus its functions in one block. `Counter` is the type, `c` is one object of that type.

### `public` / `private`
Access levels that say who can use something.
- `private int value;` only code inside the class can use it.
- `public void add(...)` any code can call it.
- `public class Counter` must live in a file named `Counter.java`.

Usual pattern: variables `private`, methods `public`.

### `void`
Return type, same idea as C. `void add(int n)` returns nothing, `int getValue()` returns an `int`.

### `public static void main(String[] args)`
Java's `int main(void)`.

| Word | Meaning |
|---|---|
| `public` | the Java launcher must be able to call it |
| `static` | belongs to the class, not an object (no object exists yet when the program starts) |
| `void` | returns nothing (no `return 0;`) |
| `main` | fixed name Java looks for |
| `String[] args` | command-line arguments, like `char *argv[]` in C |

`args[0]` is the first argument (not the program name, unlike C). There is no `argc`: use `args.length`.

### `new`
`new Counter()` is like `malloc` plus initialization: it allocates the object and runs its constructor. It returns a reference (a pointer you never dereference with `*`, and never `free`).

```c
Counter *c = malloc(sizeof(Counter));   // C
```
```java
Counter c = new Counter();              // Java
```

Quick demo of `String[] args`:

In [11]:
%%writefile Args.java
public class Args {
    public static void main(String[] args) {
        System.out.println("count: " + args.length);
        for (String a : args)
            System.out.println("arg: " + a);
    }
}

Writing Args.java


In [12]:
!java Args.java hello world 42

count: 3
arg: hello
arg: world
arg: 42


## 4. Summary table

| Concept | C | Java |
|---|---|---|
| Data type | `struct` | `class` |
| Functions on data | separate functions | methods inside the class |
| Hide data | not possible | `private` |
| Polymorphism | function pointers | `extends` + `@Override` |
| Memory | `malloc` / `free` | `new` + garbage collector |
| Errors | return codes (`-1`) | exceptions (`throw` / `try-catch`) |
| Entry point | `int main(int argc, char *argv[])` | `public static void main(String[] args)` |

Mental shortcut: a Java class is roughly a C `struct` + its functions + a function-pointer table, with the compiler enforcing the rules.

## 5. Exercises

1. Add `deposit` to the `Counter` class in C and Java. Compare code size.
2. Add a `Triangle` to both shapes programs. In Java, only a new class is needed; the `main` loop stays unchanged.
3. Change `private` to `public` on `value` and set it from `main`. What changes?
4. Print `args.length` for a run with no arguments.

**Next topic:** interfaces vs abstract classes.